# whisper-small LoRA 파인튜닝 — 608 스모크 검증

**목적**: 파이프라인이 Colab T4에서 에러 없이 도는지 확인(소량 2화자 데이터).
학습 품질은 무의미 — "노트북이 끝까지 돈다"만 본다.

**전제**: 런타임 → 런타임 유형 변경 → **T4 GPU**. 그리고 `colab_trainset.zip`을 드라이브에 업로드.


## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (torchao 제거 + numpy 고정) → 실행 후 [런타임 → 세션 다시 시작] 필수
numpy 버전을 **손으로 적지 않는다**. Colab 이미지의 torch/librosa는 그 이미지의 numpy에 맞춰 빌드돼 있으므로,
지금 런타임의 numpy 버전을 읽어서 그대로 못박는다. 재시작 후에는 셀 2를 건너뛰고 **셀 2.5**부터.

> ⚠️ 셀 2는 **초기화된 런타임**에서 실행해야 한다. 이미 `pip -U`를 한 번 돌린 세션이면 numpy가 이미 올라가 있어서
> 잘못된 버전을 고정하게 된다 → 먼저 [런타임 → 세션 삭제 후 다시 시작].


In [ ]:
# Colab 이미지가 들고 있는 numpy 버전을 읽어서 그대로 고정한다.
# 숫자를 하드코딩하면(예전의 numpy==2.0.2) Colab 이미지가 바뀔 때마다 다시 깨진다:
#   numpy가 너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   numpy가 너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정")

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U transformers datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

!pip show numpy | grep -i ^Version    # 디스크에 실제로 깔린 버전 = NPV여야 정상
print("✅ 설치 완료 → [런타임 → 세션 다시 시작] 반드시! → 셀 2.5부터")

## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + 업로드한 zip 풀기 + 경로 확인
`ZIP`을 본인이 올린 위치로 수정.

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset_big6.zip"   # ← 업로드한 zip 경로(현재 6차: 재정렬본)
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())
for f in ["train.jsonl","test.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + LoRA

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

# 베이스 모델. zero-shot 비교(셀 10) 결과로 large-v3-turbo를 채택했다.
#
#   모델              단어 CER   문장 CER
#   small               2.000      0.404
#   medium              1.376      0.363
#   large-v3-turbo      0.890      0.321   ← 두 지표 모두 승
#
# 단어가 모델 크기에 훨씬 민감하다(2.2배 개선 vs 문장 1.26배). 앱 최약점이
# 단어 수준 구음장애라 이 차이가 그대로 제품 성능이다. turbo는 디코더가 얕아
# large-v3(1550M)보다 훨씬 빠르면서 T4에 들어간다.
#
# 되돌리려면 이 한 줄만 바꾼다. 프로세서도 같은 ID로 로드되므로 mel 빈 수
# (large-v3 계열 128 vs small/medium 80)가 자동으로 맞는다 — 프로세서를
# 재사용하면 입력이 조용히 깨진다.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# **fp32로 명시 로드해야 한다.** large-v3-turbo는 config에 torch_dtype: float16이
# 박혀 있어 최신 transformers가 fp16 가중치로 올린다. 그러면 특징 추출기가 내놓는
# fp32 입력과 어긋나 인코더 conv에서 터진다:
#   RuntimeError: Input type (float) and bias type (c10::Half) should be the same
# small은 config가 fp32라 이 문제가 없어 지금까지 안 드러났다.
#
# 혼합정밀은 Trainer의 fp16=True(autocast)가 맡는다. 가중치까지 fp16으로 두고
# 학습하면 마스터 가중치가 없어 불안정하다. T4는 Turing이라 bf16이 없어 이 조합이
# 유일한 선택지다.
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj","v_proj"], lora_dropout=0.05))
model.print_trainable_parameters()

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제
model.generation_config.max_new_tokens = 100


## 셀 5 — 데이터셋 로드 + 전처리

In [ ]:
from datasets import load_dataset, Audio
ds = load_dataset("json", data_files={
    "train": f"{BASE}/train.jsonl", "test": f"{BASE}/test.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

# task_type(단어/문장)이 있으면 매핑으로 사라지기 전에 보존(과제유형별 CER용).
# 구 배치 zip은 이 필드가 없을 수 있어, 없으면 None으로 두고 셀 9에서 건너뛴다.
test_task_types = (
    list(ds["test"]["task_type"]) if "task_type" in ds["test"].column_names else None
)

def prep(b):
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b
ds = ds.map(prep, remove_columns=ds["train"].column_names)
print(ds)

## 셀 7.5 — 앱 채점기 로드 (제품과 같은 잣대)

608 평가의 표준 CER은 **앱 조건보다 비관적**이다. 앱은 단어 과제에 음소 유사 가중
거리를 써서 조음 유사 혼동(ㅂ↔ㅍ, 종성 탈락)을 부분 오류로 본다. 같은 채점기로
재야 "이 모델을 붙이면 앱이 얼마나 좋아지나"에 답할 수 있다.

무엇보다 앱이 내는 판정은 오류율이 아니라 **임계값 통과 여부**(≤0.34)다.
평균이 경계 아래인 것과 항목별 통과율은 다른 이야기다.


In [ ]:
# scripts/asr_eval/app_scorer.py 를 Colab으로 가져온다.
# 리포를 통째로 올리기 번거로우면 이 셀에 붙여넣어도 되지만, 원본이 바뀌면
# 어긋나므로 파일을 올리는 쪽을 권한다.
#
# 원본(단일 진실): frontend/src/memory-link/patient/quiz/domain/phoneticDistance.ts
#                  frontend/src/memory-link/patient/quiz/domain/speechScore.ts
# 파이썬 이식본: scripts/asr_eval/app_scorer.py (test_app_scorer.py가 두 구현을 고정)

import os, sys, urllib.request

APP_SCORER = "/content/app_scorer.py"
if not os.path.exists(APP_SCORER):
    # (a) 드라이브에 올려 뒀으면 그걸 쓴다
    cand = "/content/drive/MyDrive/app_scorer.py"
    if os.path.exists(cand):
        import shutil; shutil.copy(cand, APP_SCORER)
    else:
        raise SystemExit(
            "app_scorer.py가 없다. scripts/asr_eval/app_scorer.py를 드라이브"
            " 루트에 올리거나 /content/에 업로드하라."
        )
sys.path.insert(0, "/content")
import app_scorer
print("앱 채점기 로드 · 정답 임계값:", app_scorer.SPEECH_PASS_THRESHOLD)


## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 (baseline 0.70 비교)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")
def compute_metrics(p):
    lab = p.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    pred = processor.tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    ref  = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    return {"cer": cer_metric.compute(predictions=pred, references=ref)}

## 셀 8 — 학습
- `learning_rate=1e-4` (1e-3은 과학습/발산). 데이터 많아지면 `num_train_epochs`는 3~5로.
- 실제(대규모) 학습은 `output_dir`을 드라이브로(체크포인트 재개).

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-608-lora",    # 대규모 학습은 /content/drive/... 로
    # large-v3-turbo(809M)는 small(244M)보다 크다. T4 16GB에서 배치 8은 위험해
    # 4로 낮추고 누적 2로 유효 배치 8을 유지한다(학습 동역학은 그대로).
    # OOM이 나면 배치 2 · 누적 4, 그래도 안 되면 gradient_checkpointing=True.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4, warmup_steps=20,       # LoRA whisper 표준 LR
    # small은 1에폭에 바닥을 치고 이후 나빠졌다(5차 1ep 0.292 → 4ep 0.319).
    # 그런데 large-v3-turbo는 **2에폭이 최고**였다(6차) — 곡선이 아직 안 꺾였다는
    # 뜻이라 더 내려갈 여지가 있다. 아래 load_best_model_at_end가 과학습을 막아
    # 주므로 에폭을 늘리는 건 안전하다: 나빠지면 이전 체크포인트가 복원되고
    # 시간만 더 쓴다.
    num_train_epochs=4, fp16=True,
    eval_strategy="epoch", logging_steps=10,
    # val loss는 끝까지 내려가는데 CER은 아니라(지표 갈라짐) loss 기준으로는
    # 최고를 못 고른다. CER을 기준으로 최고 체크포인트를 복원한다.
    # 이게 없어 5차에서 0.292 대신 0.319를 저장했다(−8% 손실).
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="cer", greater_is_better=False,
    predict_with_generate=True, generation_max_length=225,
    remove_unused_columns=False, label_names=["labels"], report_to="none")
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor)

# 공정 비교: 파인튜닝 전 baseline(=원본 whisper) CER 먼저 찍고 학습.
print("파인튜닝 전 baseline:", trainer.evaluate())
trainer.train()
# load_best_model_at_end로 복원된 최고 체크포인트의 실제 성적.
print("최종(최고 체크포인트) CER:", trainer.evaluate())


## 셀 9 — 결과 + 파이프라인 OK 확인

In [ ]:
m = trainer.evaluate()
print(f"파인튜닝 후 전체 CER {m['eval_cer']:.3f}  (참고용 — 판단은 아래 두 지표로)")

# 앱의 두 기능에 각각 대응하는 두 지표. 섞은 전체 CER로 판단하지 않는다 —
# CER은 글자 수 가중이라 이 test에선 93%가 문장으로 결정돼 단어가 안 보인다.
#
#   단어 정확 일치율  →  QAB 검사(그림 이름대기·단어 따라말하기)
#   문장 CER          →  대화 모드, 따라말하기·읽기
#
# 단어에 CER 대신 정확도를 쓰는 이유: 참조가 평균 1.4자라 조금만 더 뱉어도 CER이
# 폭발한다(1글자 정답에 5글자면 CER 5.0). "이 음절을 맞게 인식했나"가 앱이 묻는
# 질문이다.
if test_task_types is not None:
    import re, unicodedata

    def norm(t):
        t = unicodedata.normalize("NFC", t or "")
        t = re.sub(r"[^\w가-힣\s]", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    pred = trainer.predict(ds["test"])
    lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyps = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)

    w = [i for i, t in enumerate(test_task_types) if t == "wordlist"]
    s = [i for i, t in enumerate(test_task_types) if t == "narrative"]
    if w:
        acc = sum(norm(hyps[i]) == norm(refs[i]) for i in w) / len(w)
        c = cer_metric.compute(predictions=[hyps[i] for i in w],
                               references=[refs[i] for i in w])
        print(f"  [검사]   단어 정확도 {acc:.1%} · CER {c:.3f}  (n={len(w)})")
    if s:
        c = cer_metric.compute(predictions=[hyps[i] for i in s],
                               references=[refs[i] for i in s])
        print(f"  [대화]   문장 CER {c:.3f}  (n={len(s)})")
        print("           ※ 대본 낭독이라 자유 발화 대화의 낙관적 대리치다.")

    # 앱 잣대 — 위 CER은 표준 편집거리라 앱 조건보다 비관적이다. 앱은 단어에
    # 음소 유사 가중 거리를 쓰고, 실제 판정은 임계값 통과 여부다.
    # **통과율이 제품에 직결되는 숫자**다(평균이 경계 아래여도 통과율은 100%가 아니다).
    try:
        import app_scorer
    except ImportError:
        print("  (앱 채점기 미로드 — 셀 7.5를 먼저 실행하면 앱 잣대 통과율이 나온다)")
    else:
        print()
        print("  ── 앱 잣대(phoneticEditDistance, 정답 경계 0.34) ──")
        if w:
            r = app_scorer.score_batch([hyps[i] for i in w], [refs[i] for i in w], "word")
            print(f"  [검사]   통과율 {r['pass_rate']:.1%} · 평균 오류율 "
                  f"{r['error_rate']:.3f}  (n={r['n']})")
        if s:
            r = app_scorer.score_batch([hyps[i] for i in s], [refs[i] for i in s], "sentence")
            print(f"  [대화]   통과율 {r['pass_rate']:.1%} · 평균 오류율 "
                  f"{r['error_rate']:.3f}  (n={r['n']})")
else:
    print("  (이 zip은 task_type 필드가 없는 구 배치 — 두 지표 분리 불가)")

# 저장(LoRA 어댑터만 수십MB)
model.save_pretrained(f"{BASE}/whisper-608-lora-final")


## 셀 10 — 모델 크기 zero-shot 비교 (학습 없음)

**무엇을 정하려는 비교인가**: 파인튜닝 베이스 모델을 small에서 키울지 여부.

판단 기준을 미리 정해 둔다(수치를 보고 정하면 자기합리화가 된다).

| 관측 | 결론 |
|---|---|
| large zero-shot이 파인튜닝한 small보다 좋음 | 베이스 교체가 1순위 |
| large가 **단어에서만** 크게 좋음 | 단어는 용량 문제 → 큰 모델 + 단어 데이터 |
| 모든 모델이 단어에서 비슷하게 나쁨 | 용량이 아니라 도메인(구음장애) 문제 → 데이터가 계속 레버 |
| medium이 large-turbo와 비슷 | 비용상 medium 채택 |


In [ ]:
# 앱의 두 기능에 각각 대응하는 **두 지표**를 따로 낸다. 섞지 않는다.
#
#   단어 정확 일치율  →  QAB 검사(그림 이름대기·단어 따라말하기)
#   문장 CER          →  대화 모드, 따라말하기·읽기
#
# 왜 섞으면 안 되나: CER은 글자 수로 가중되는 마이크로평균인데 이 test는 단어
# 참조가 210자, 문장이 2807자다. 전체 CER의 93%가 문장으로 결정돼 단어 성능이
# 사실상 안 보인다. 그렇다고 문장이 덜 중요한 것도 아니라(대화 모드), 둘 다
# 1급 지표로 두고 각각 보고한다.
#
# 단어에 CER 대신 정확 일치율을 쓰는 이유: 참조가 평균 1.4자라 조금만 더 뱉어도
# CER이 폭발한다(1글자 정답에 5글자를 뱉으면 CER 5.0). "이 음절을 맞게 인식했나"가
# 앱이 실제로 묻는 질문이고, 정확도 34%가 CER 0.890보다 해석 가능하다.
#
# 단어 클립에는 max_new_tokens를 좁힌다. 0.58초 클립에 100토큰을 허용하면 whisper가
# 문장을 지어내고, 그 환각이 수치를 지배한다. 모델 전부에 동일 적용해야 비교가 산다.

import gc, torch
from collections import Counter
from datasets import load_dataset, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

ZERO_SHOT_MODELS = [
    "openai/whisper-small",            # 현재 베이스
    "openai/whisper-medium",
    "openai/whisper-large-v3-turbo",
]
BATCH = 8
WORD_MAX_TOKENS = 8      # 단어 클립: 환각 억제
SENT_MAX_TOKENS = 100    # 문장 클립: 기존과 동일

raw = load_dataset("json", data_files={"test": f"{BASE}/test.jsonl"})["test"]
raw = raw.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
raw = raw.cast_column("audio", Audio(sampling_rate=16000))
if "task_type" not in raw.column_names:
    raise SystemExit("이 zip에는 task_type이 없다 — 5차 이후 패키지를 써라.")
task_types = list(raw["task_type"])
refs_all = list(raw["text"])
w_idx = [i for i, t in enumerate(task_types) if t == "wordlist"]
s_idx = [i for i, t in enumerate(task_types) if t == "narrative"]
print(f"test {len(raw)}개 · 단어 {len(w_idx)} · 문장 {len(s_idx)}")

def norm(t):
    """비교용 정규화 — metrics.normalize와 같은 규칙(부호 제거, 공백 축약)."""
    import re, unicodedata
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"[^\w가-힣\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def transcribe(model_id, idxs, max_new_tokens, batch=BATCH):
    proc = WhisperProcessor.from_pretrained(model_id, language="ko", task="transcribe")
    m = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16).to("cuda").eval()
    # 생성 설정은 모델 간 완전히 동일하게. 다르면 모델이 아니라 디코딩을 비교하게 된다.
    m.generation_config.language = "ko"
    m.generation_config.task = "transcribe"
    m.generation_config.no_repeat_ngram_size = 3
    m.generation_config.max_new_tokens = max_new_tokens
    m.generation_config.forced_decoder_ids = None

    hyps = []
    for i in range(0, len(idxs), batch):
        chunk = [raw[j]["audio"] for j in idxs[i : i + batch]]
        feats = proc.feature_extractor(
            [a["array"] for a in chunk], sampling_rate=16000, return_tensors="pt"
        ).input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = m.generate(feats)
        hyps.extend(proc.tokenizer.batch_decode(ids, skip_special_tokens=True))
    del m, proc
    gc.collect(); torch.cuda.empty_cache()
    return hyps

rows = []
for mid in ZERO_SHOT_MODELS:
    try:
        # 모델마다 자기 프로세서를 로드한다 — large-v3 계열은 mel 128빈, small/medium은
        # 80빈이라 재사용하면 입력이 깨진다(예외 없이 CER만 조용히 나빠진다).
        w_hyp = transcribe(mid, w_idx, WORD_MAX_TOKENS)
        s_hyp = transcribe(mid, s_idx, SENT_MAX_TOKENS)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        print(f"{mid}: OOM → 배치 2로 재시도")
        w_hyp = transcribe(mid, w_idx, WORD_MAX_TOKENS, batch=2)
        s_hyp = transcribe(mid, s_idx, SENT_MAX_TOKENS, batch=2)
    except Exception as e:                     # 한 모델이 죽어도 비교는 계속한다
        print(f"{mid}: 실패 — {type(e).__name__}: {e}")
        continue

    w_ref = [refs_all[i] for i in w_idx]
    s_ref = [refs_all[i] for i in s_idx]
    exact = sum(norm(h) == norm(r) for h, r in zip(w_hyp, w_ref)) / max(len(w_ref), 1)
    rows.append({
        "model": mid.split("/")[-1],
        "단어 정확도": exact,
        "단어 CER": cer_metric.compute(predictions=w_hyp, references=w_ref),
        "문장 CER": cer_metric.compute(predictions=s_hyp, references=s_ref),
    })
    print("  ", {k: (f"{v:.3f}" if isinstance(v, float) else v) for k, v in rows[-1].items()})

print()
print(f"{'model':<24}{'단어 정확도':>12}{'단어 CER':>10}{'문장 CER':>10}")
for r in rows:
    print(f"{r['model']:<24}{r['단어 정확도']:>11.1%}{r['단어 CER']:>10.3f}{r['문장 CER']:>10.3f}")
print()
print("※ 검사 기능은 '단어 정확도', 대화 모드는 '문장 CER'로 본다. 둘을 섞은 값은 쓰지 않는다.")
print("※ 문장은 대본 낭독이라 대화 모드의 낙관적 대리치다 — 자유 발화 평가셋은 별도 과제.")
print("※ zero-shot(학습 없음)이다. 파인튜닝한 모델은 셀 9에서 같은 두 지표로 본다.")
